In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q orjson pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 8.2 MB/s eta 0:00:00


# Library

In [ ]:
import ast
import gzip
import hashlib
import html
import json
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import orjson
import pandas as pd
import seaborn as sns
from tqdm import tqdm

tqdm.pandas()

# Utils

## Meta Data Filtering

In [ ]:
# Amazon Beauty_and_Personal_Care의 title을 기준으로 샴푸 제품 선별
INCLUDE = re.compile(r"\bshampoos?\b", re.IGNORECASE)
EXCLUDE = re.compile(r"\b(pet|dog|cat|brush|bowl|cape|dispenser|conditioner|dry shampoo)\b", re.IGNORECASE)

In [ ]:
def select_shampoo_products(metadata_path, minimum_ratings=300):
    products = []

    with gzip.open(metadata_path, "rb") as source:
        for line in source:
            product = orjson.loads(line)
            title = product.get("title") or ""
            rating_number = product.get("rating_number") or 0

            if not INCLUDE.search(title) or EXCLUDE.search(title) or rating_number < minimum_ratings:
                continue
            if not product.get("features") and not product.get("description"):
                continue

            products.append({
                "parent_asin": product["parent_asin"], "title": title, "store": product.get("store"),
                "rating_number": rating_number, "average_rating": product.get("average_rating"),
                "features": product.get("features"), "description": product.get("description"),
                "details": product.get("details"), "price": product.get("price"),
            })

    return products

## Review Data Filtering

In [ ]:
def aggregate_review_stats(review_path, candidate_parent_asin, total_review=None):
    """후보 제품별 리뷰 수, 부정·중립 리뷰 수, 인증 구매 수와 리뷰 기간을 집계한다."""
    candidate_parent_asin = set(candidate_parent_asin)
    review_count, negative_count, verified_count = Counter(), Counter(), Counter()
    min_timestamp, max_timestamp = {}, {}

    with gzip.open(review_path, "rb") as source:
        progress = tqdm(source, total=total_review, desc="리뷰 집계", unit="건", mininterval=1.0)

        for line in progress:
            review = orjson.loads(line)
            parent_asin = review.get("parent_asin")
            if parent_asin not in candidate_parent_asin:
                continue

            review_count[parent_asin] += 1
            rating = review.get("rating")
            if rating is not None and rating <= 3:
                negative_count[parent_asin] += 1
            if review.get("verified_purchase") is True:
                verified_count[parent_asin] += 1

            timestamp = review.get("timestamp")
            if timestamp is not None:
                min_timestamp[parent_asin] = min(timestamp, min_timestamp.get(parent_asin, timestamp))
                max_timestamp[parent_asin] = max(timestamp, max_timestamp.get(parent_asin, timestamp))

    return {"review_count": review_count, "negative_count": negative_count, "verified_count": verified_count,
            "min_timestamp": min_timestamp, "max_timestamp": max_timestamp}

In [ ]:
def load_eligible_reviews(review_path, eligible_products):
    eligible_ids, reviews = set(eligible_products["parent_asin"].dropna()), []

    with gzip.open(review_path, "rb") as source:
        for line in tqdm(source, desc="선정 제품 리뷰 추출", unit="건"):
            review = orjson.loads(line)
            if review.get("parent_asin") not in eligible_ids:
                continue

            reviews.append({
                "parent_asin": review.get("parent_asin"), "asin": review.get("asin"), "user_id": review.get("user_id"),
                "rating": review.get("rating"), "review_title": review.get("title"), "review_text": review.get("text"),
                "timestamp": review.get("timestamp"), "helpful_vote": review.get("helpful_vote"),
                "verified_purchase": review.get("verified_purchase"),
            })

    reviews_df = pd.DataFrame(reviews)
    print(f"제품: {len(eligible_ids):,}개")
    print(f"리뷰: {len(reviews_df):,}건")
    return reviews_df

## Prepare RDS dataset

In [ ]:
def prepare_rds_products(df):
    return df.drop(columns="index", errors="ignore").drop_duplicates("parent_asin").copy()


def prepare_rds_reviews(df):
    reviews = df.drop(columns="index", errors="ignore").copy()
    reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
    reviews["timestamp"] = pd.to_numeric(reviews["timestamp"], errors="coerce")
    reviews["helpful_vote"] = pd.to_numeric(reviews["helpful_vote"], errors="coerce").fillna(0).astype("int64")
    reviews["verified_purchase"] = reviews["verified_purchase"].astype(str).str.lower().map({"true": True, "false": False}).fillna(False)
    reviews = reviews.drop_duplicates(subset=["parent_asin", "asin", "user_id", "timestamp", "review_text"])
    return reviews.reset_index(drop=True)


def filter_single_asin_products(products_df, reviews_df):
    asin_stats = reviews_df.groupby("parent_asin").agg(asin_count=("asin", "nunique"), analyzed_asin=("asin", "first")).reset_index()
    single_asin = asin_stats[asin_stats["asin_count"] == 1][["parent_asin", "analyzed_asin"]].copy()
    products = products_df.merge(single_asin, on="parent_asin", how="inner", validate="one_to_one")
    reviews = reviews_df.merge(single_asin, left_on=["parent_asin", "asin"], right_on=["parent_asin", "analyzed_asin"], how="inner", validate="many_to_one")
    return products.reset_index(drop=True), reviews.reset_index(drop=True)


def rebuild_product_stats(products_df, reviews_df):
    stats = reviews_df.groupby("parent_asin").agg(review_count=("parent_asin", "size"), negative_count=("rating", lambda x: x.le(3).sum()), verified_count=("verified_purchase", "sum"), min_timestamp=("timestamp", "min"), max_timestamp=("timestamp", "max")).reset_index()
    stat_columns = ["review_count", "negative_count", "verified_count", "min_timestamp", "max_timestamp"]
    return products_df.drop(columns=stat_columns, errors="ignore").merge(stats, on="parent_asin", how="inner", validate="one_to_one")


def validate_final_data(products_df, reviews_df):
    actual = reviews_df.groupby("parent_asin").agg(actual_review_count=("parent_asin", "size"), actual_negative_count=("rating", lambda x: x.le(3).sum()), actual_verified_count=("verified_purchase", "sum"), actual_asin_count=("asin", "nunique")).reset_index()
    check = products_df.merge(actual, on="parent_asin", how="left", validate="one_to_one")
    results = {
        "review_count_mismatch": check["review_count"].ne(check["actual_review_count"]).sum(),
        "negative_count_mismatch": check["negative_count"].ne(check["actual_negative_count"]).sum(),
        "verified_count_mismatch": check["verified_count"].ne(check["actual_verified_count"]).sum(),
        "multiple_asin_products": check["actual_asin_count"].ne(1).sum(),
        "wrong_asin_reviews": reviews_df["asin"].ne(reviews_df["analyzed_asin"]).sum(),
    }
    return pd.Series(results, name="count")

# Prepare RAG dataset

In [ ]:
def make_product_rag(products):
    columns = ["parent_asin", "analyzed_asin", "title", "store", "features", "description", "price"]
    rag = products[columns].copy()

    for column in ["title", "store", "features", "description"]:
        rag[column] = rag[column].fillna("").astype(str).apply(html.unescape)

    rag["document_id"] = "product:" + rag["analyzed_asin"]
    rag["document_type"] = "product"
    rag["text"] = "Product: " + rag["title"] + "\nASIN: " + rag["analyzed_asin"] + "\nBrand: " + rag["store"] + "\nFeatures: " + rag["features"] + "\nDescription: " + rag["description"]
    rag = rag.drop_duplicates("document_id")
    return rag[["document_id", "document_type", "parent_asin", "analyzed_asin", "text", "store", "price"]]

In [ ]:
def make_review_rag(reviews):
    columns = ["parent_asin", "analyzed_asin", "asin", "user_id", "rating", "review_title", "review_text", "timestamp", "helpful_vote", "verified_purchase"]
    rag = reviews[columns].copy()

    for column in ["review_title", "review_text"]:
        rag[column] = rag[column].fillna("").astype(str).apply(html.unescape)

    rag = rag[rag["review_text"].str.strip().ne("")].drop_duplicates(subset=["parent_asin", "asin", "user_id", "timestamp", "review_text"]).copy()
    rag["review_id"] = rag.apply(lambda row: hashlib.sha256(f"{row['parent_asin']}|{row['asin']}|{row['user_id']}|{row['timestamp']}|{row['review_text']}".encode()).hexdigest()[:16], axis=1)
    rag["document_id"] = "review:" + rag["review_id"]
    rag["document_type"] = "review"
    rag["text"] = "Title: " + rag["review_title"] + "\nReview: " + rag["review_text"]
    return rag[["document_id", "document_type", "review_id", "parent_asin", "analyzed_asin", "asin", "text", "rating", "timestamp", "helpful_vote", "verified_purchase"]]

# Deal with asin column mismatch

In [ ]:
# filter_single_asin_products()로 대체됨.
# parent_asin 아래 실제 리뷰 ASIN이 하나인 제품만 유지해 옵션 간 속성 혼합을 방지한다.

# RDS Data

## Load Meta Dataset

In [ ]:
metadata_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Meta/meta_Beauty_and_Personal_Care.jsonl.gz"
products_df = pd.DataFrame(select_shampoo_products(metadata_path))
print("조건 통과 제품:", len(products_df))

조건 통과 제품: 2093


In [ ]:
products_df.head()

,parent_asin,title,store,rating_number,average_rating,features,description,details,price
0,B06WGN456Y,"Axe Shampoo, Peace 12 oz",AXE,518,4.5,[Axe Peace Shampoo cleanses guy’s hair with a ...,[Every masterpiece starts with a clean canvas....,"{'Brand': 'AXE', 'Item Form': 'Liquid', 'Scent...",11.99
1,B08HGR6FR2,24 Pieces Empty Bottles with Flip Cap Travel S...,Boao,758,4.7,[Package content: you will receive totally 24 ...,[],"{'Brand': 'Boao', 'Material': 'Plastic', 'Colo...",10.99
2,B01NBW4ANA,Artnaturals Tea Tree Shampoo - (16 Fl Oz / 473...,Artnaturals,1208,4.5,[TEA TREE OIL: ArtNaturals tea tree shampoo co...,[],"{'Brand': 'Artnaturals', 'Item Form': 'Liquid'...",12.95
3,B07HZ1QMVP,TBH Teen and Kids Shampoo - Tween and Teen Sha...,Fairy Tales,1062,4.2,[Teen and Kid Shampoo for Girls and Boys : Kid...,[],"{'Brand': 'Fairy Tales', 'Item Form': 'Liquid'...",9.95
4,B0BHX2XV4R,Tigi Bed Head Urban Anti+dotes Recovery Shampo...,Bed Head,1236,4.7,"[Produced with the highest grade materials, Sa...",[],"{'Brand': 'TIGI', 'Item Form': 'Liquid', 'Hair...",NaN


In [ ]:
candidate_parent_asin = set(products_df["parent_asin"])
review_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Review/Beauty_and_Personal_Care.jsonl.gz"
stats = aggregate_review_stats(review_path, candidate_parent_asin)

리뷰 집계: 23911390건 [01:29, 265727.11건/s]


In [ ]:
stat_dict = {"review_count": stats["review_count"], "negative_count": stats["negative_count"], "verified_count": stats["verified_count"], "min_timestamp": stats["min_timestamp"], "max_timestamp": stats["max_timestamp"]}

for column, mapping in stat_dict.items():
    products_df[column] = products_df["parent_asin"].map(mapping)
    if column.endswith("_count"):
        products_df[column] = products_df[column].fillna(0).astype("int64")

In [ ]:
has_product_text = products_df["features"].str.len().gt(0) | products_df["description"].str.len().gt(0)
eligible_products = products_df[(products_df["review_count"] >= 50) & (products_df["negative_count"] >= 10) & has_product_text].copy()

for column in ["features", "description"]:
    eligible_products[column] = eligible_products[column].apply(lambda x: " ".join(map(str, x)) if isinstance(x, list) else x)

In [ ]:
eligible_ids = set(eligible_products["parent_asin"])
total_reviews = sum(stats["review_count"][parent_asin] for parent_asin in eligible_ids)
total_negative_reviews = sum(stats["negative_count"][parent_asin] for parent_asin in eligible_ids)

print(f"선정 제품: {len(eligible_ids):,}개")
print(f"전체 리뷰: {total_reviews:,}건")
print(f"부정·중립 리뷰: {total_negative_reviews:,}건")

선정 제품: 967개
전체 리뷰: 237,498건
부정·중립 리뷰: 57,975건


## Load User Review Dataset


```python
# RDS에 원본 텍스트가 필요한 이유

- 벡터 DB를 다시 구축할 수 있는 원본 데이터가 됨
- 검색 결과의 원문을 정확히 조회할 수 있음
- SQL 조건으로 리뷰를 먼저 필터링할 수 있음
- 전처리·임베딩 모델·문서 형식이 변경돼도 다시 생성할 수 있음
- 대시보드에서 리뷰 원문과 제품 설명을 표시할 수 있음
- 벡터 DB의 데이터가 손상되거나 삭제돼도 복구할 수 있음
```

In [ ]:
eligible_products = eligible_products.drop_duplicates("parent_asin").copy()
reviews_df = load_eligible_reviews(review_path, eligible_products)

선정 제품 리뷰 추출: 23911390건 [01:18, 305468.09건/s]


제품: 967개
리뷰: 237,498건


In [ ]:
reviews_df.head()

,parent_asin,asin,user_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase
0,B0BHP2ZS4G,B006BLON64,AGKHLEW2SOWHNMFQIJGBECAF7INQ,5.0,This stuff is a luxurious vacation in a bottle.,It smells heavenly but not overwhelming at all...,1588615673460,2,True
1,B07NNNVMVV,B07NNNVMVV,AFSKPY37N3C43SOI5IEXEK5JSIYA,5.0,Great tear free shampoo!,"Great kids shampoo. Leaves hair clean, fresh s...",1565519150437,1,False
2,B077XW2LKS,B077XW2LKS,AHGAOIZVODNHYMNCBV4DECZH42UQ,5.0,OK as long as its OK from being liquid and tur...,its really hot now in NY so it showed up liqu...,1531595936905,0,True
3,B0BHX3VWNG,B003M04GZK,AHGAOIZVODNHYMNCBV4DECZH42UQ,5.0,LOVE IT,always loved prell hard to find in stores bu...,1487361962000,2,True
4,B000764XFK,B000764XFK,AEDJZTFJDNOW6JDLMBZNK2QBN3GQ,5.0,Awesome stuff,I have used many different shampoos and condit...,1481884210000,0,True


In [ ]:
products_rds = prepare_rds_products(eligible_products)
reviews_rds = prepare_rds_reviews(reviews_df)
print(f"중복 제거 후 리뷰: {len(reviews_rds):,}건")

중복 제거 후 리뷰: 235,175건


In [ ]:
# 개별 제품 속성이 섞이지 않도록 parent_asin당 실제 리뷰 ASIN이 하나인 제품만 유지
products_rds, reviews_rds = filter_single_asin_products(products_rds, reviews_rds)

# 중복 제거와 단일 ASIN 필터가 모두 끝난 리뷰를 기준으로 제품 통계 재계산
products_rds = rebuild_product_stats(products_rds, reviews_rds)

In [ ]:
has_product_text = products_rds["features"].fillna("").astype(str).str.strip().ne("") | products_rds["description"].fillna("").astype(str).str.strip().ne("")
products_rds = products_rds[(products_rds["review_count"] >= 50) & (products_rds["negative_count"] >= 10) & has_product_text].copy()
reviews_rds = reviews_rds[reviews_rds["parent_asin"].isin(products_rds["parent_asin"])].copy()

# 최종 제품 필터 후 통계를 한 번 더 동기화
products_rds = rebuild_product_stats(products_rds, reviews_rds).reset_index(drop=True)
reviews_rds = reviews_rds.reset_index(drop=True)

print(f"선정 제품: {len(products_rds):,}개")
print(f"전체 리뷰: {len(reviews_rds):,}건")
print(f"부정·중립 리뷰: {(reviews_rds['rating'] <= 3).sum():,}건")

선정 제품: 547개
전체 리뷰: 105,060건
부정·중립 리뷰: 25,152건


In [ ]:
validation_result = validate_final_data(products_rds, reviews_rds)
display(validation_result)
assert validation_result.eq(0).all(), "최종 데이터 검증 실패"

,count
review_count_mismatch,0
negative_count_mismatch,0
verified_count_mismatch,0
multiple_asin_products,0
wrong_asin_reviews,0


In [ ]:
products_rds.to_parquet("/content/drive/MyDrive/products_rds.parquet", index=False)
reviews_rds.to_parquet("/content/drive/MyDrive/reviews_rds.parquet", index=False)

In [ ]:
empty_review_text = (
    reviews_rds["review_text"].isna()
    | reviews_rds["review_text"].fillna("").str.strip().eq("")
).sum()

empty_rating = reviews_rds["rating"].isna().sum()

# Parquet 원본에서는 reviewed_at이 아니라 timestamp 컬럼
empty_reviewed_at = reviews_rds["timestamp"].isna().sum()

print("빈 리뷰 본문:", empty_review_text)
print("빈 평점:", empty_rating)
print("빈 리뷰 작성 시각:", empty_reviewed_at)

빈 리뷰 본문: 95
빈 평점: 0
빈 리뷰 작성 시각: 0


# RAG Data

In [ ]:
products_rag = make_product_rag(products_rds)
reviews_rag = make_review_rag(reviews_rds)
print(f"제품 문서: {len(products_rag):,}건, 리뷰 문서: {len(reviews_rag):,}건")

제품 문서: 547건, 리뷰 문서: 104,965건


In [ ]:
products_rag.loc[0, "text"]

'Product: Artnaturals Tea Tree Shampoo - (16 Fl Oz / 473ml) - Sulfate Free – Made with 100% Pure Therapeutic Grade Tea Tree Essential Oil\nASIN: B01NBW4ANA\nBrand: Artnaturals\nFeatures: TEA TREE OIL: ArtNaturals tea tree shampoo contains 100% pure tea tree essential oil. DANDRUFF & ITCHY SCALP: The anti:itching, anti:dandruff benefits of borage oil join forces with watercress extract, which keeps scalp moisturized, preventing dry:scalp triggered dandruff and flakiness. HEAD LICE: There has been many studies showing that tea tree oil may be able to kill lice in the nymph and adult stages as well as reduce the number of eggs that will hatch. HAIR GROWTH: Our Shampoo stimulates new hair growth by clearing out dead skin cells from hair follicles. Scalp will be deep cleansed, and hair will be super:shiny and bouncy. SATISFACTION GUARANTEED : ArtNaturals offers a 100% satisfaction guaranteed or your money back. Try it risk free with no questions asked and top customer service. You have noth

In [ ]:
reviews_rag.loc[0, "text"]

'Title: OK as long as its OK from being liquid and turning back into solid\nReview: its really hot now in NY  so it showed up liquid .... but returned to solid in my house .. so im hoping its OK  otherwise im impressed with the size  i had NO clue it was sooo big'

In [ ]:
products_rag.to_json("/content/drive/MyDrive/products_rag.jsonl", orient="records", lines=True, force_ascii=False)
reviews_rag.to_json("/content/drive/MyDrive/reviews_rag.jsonl", orient="records", lines=True, force_ascii=False)